In [13]:

from HPS_regular import RegularHeatPumpStudy
from HeatPumpStudy import HeatPumpStudy
from read_csv import read_energy_mix_csv, read_hdd_csv
import pandas as pd
%load_ext autoreload
%autoreload 2

# Assumptions:
- no water heating
- no cooling
- temperature over the whole month is constant
- heat consumption is constant (heat pump is running all the time)
- COP is constant over the whole month
- no losses due to variable speed
- minimum temperature lift of 20°C for the heatpump
- Heat pump with the ideal required capacity is available 

In [14]:
working_fluid="R290" # Propane

# Monthly energy mix in Germany in 2022
# Source: https://www.energy-charts.info/charts/energy/chart.htm?l=de&c=DE&chartColumnSorting=default&source=total&stacking=stacked_percent&partsum=1&interval=month&month=-1&year=2022
file_path = "source_data/energy_mix_2022.csv"
monthly_energy_mix = read_energy_mix_csv(file_path)

# Monthly Heating degree days in germany (in Celsius)
# Source: https://ec.europa.eu/eurostat/databrowser/view/NRG_CHDD_M__custom_6030336/default/table?lang=en
file_path = 'source_data/monthly_hdd_18-22.csv'
monthly_hdd = read_hdd_csv(file_path)

# Monthly average ambient temperatures in Germany (in Celsius)
# Source: https://www.timeanddate.com/weather/germany/berlin/climate
monthly_ambient_temps = [1, 2, 5, 9, 14, 17, 19, 18, 14, 10, 5, 2]

electricity_cost = 0.35 # prognosis until 2040 in €/kWh_el source: https://www.zfk.de/politik/deutschland/strompreis-prognose-2042-habeck-ministerium#:~:text=Strompreis%3A%2037%20bis%2042%20Cent,pro%20kWh%20im%20Jahr%202042.
electricty_cost_no_subvention = 0.50 # in €/kWh_el source: https://www.bmwk.de/Redaktion/DE/Schlaglichter-der-Wirtschaftspolitik/2023/01/03-im-fokus.html#:~:text=F%C3%BCr%20private%20Haushalte%20und%20Unternehmen%20mit%20einem%20Gasverbrauch%20bis%20zu,%2C5%20Cent%20pro%20kWh).
gas_cost = 0.114 # in €/kWh_th source: https://www.destatis.de/DE/Themen/Wirtschaft/Preise/Erdgas-Strom-DurchschnittsPreise/_inhalt.html
gas_cost_no_subvention = 0.22 # in €/kWh_th source: https://www.bmwk.de/Redaktion/DE/Schlaglichter-der-Wirtschaftspolitik/2023/01/03-im-fokus.html#:~:text=F%C3%BCr%20private%20Haushalte%20und%20Unternehmen%20mit%20einem%20Gasverbrauch%20bis%20zu,%2C5%20Cent%20pro%20kWh).

# CO2 emissions per kWh for different energy sources (in kg CO2/kWh_el)
# Assumptions:
coal_co2_emissions = 0.97 #   Source: https://www.volker-quaschning.de/datserv/CO2-spez/index_e.php
natural_gas_co2_emissions = 0.43 #   Source: https://www.volker-quaschning.de/datserv/CO2-spez/index_e.php
nuclear_co2_emissions = 0.012 #   Source: https://www.ipcc.ch/site/assets/uploads/2018/02/ipcc_wg3_ar5_annex-iii.pdf (page 1335)
renewable_co2_emissions = 0.04 #   Source: https://www.ipcc.ch/site/assets/uploads/2018/02/ipcc_wg3_ar5_annex-iii.pdf (page 133)

# TODO:improvement replace assumptions about heating requirements and different insulation classes with more granular data from https://www.ise.fraunhofer.de/content/dam/ise/de/downloads/pdf/Forschungsprojekte/BMWi-03ET1272A-WPsmart_im_Bestand-Schlussbericht.pdf

# median living space area that needs heating in Germany (in m²) source: https://www.ise.fraunhofer.de/content/dam/ise/de/downloads/pdf/Forschungsprojekte/BMWi-03ET1272A-WPsmart_im_Bestand-Schlussbericht.pdf
heating_area = 183
# Heating demand for new, well-insulated homes (in kWh/m²/year) and assuming 200m² living space per heat pump
# Source: https://www.thermondo.de/info/rat/heizen/heizwaermebedarf-ermitteln/
specific_new_home_heating_demand = 30

# median Heating demand for existing homes 
# Source:  https://www.ise.fraunhofer.de/content/dam/ise/de/downloads/pdf/Forschungsprojekte/BMWi-03ET1272A-WPsmart_im_Bestand-Schlussbericht.pdf
specific_stock_home_heating_demand = 110

new_home_heating_demand = specific_new_home_heating_demand * heating_area
stock_home_heating_demand = specific_stock_home_heating_demand * heating_area

# Heating water temperature for new homes with large radiators (in Celsius)
new_home_heating_temp =  40

# Heating water temperature for stock_home with small radiators (in Celsius)
stock_home_heating_temp = 65

renovated_heating_temp = 45

# Base temperature for calculating HDD (in Celsius)
base_temperature = 18

dT_condenser = 5 # in Kelvin
dT_evaporator = 5 # in Kelvin

heatpump_investment_range = [11500,18500] # source: https://www.heizung.de/waermepumpe/luft-wasser-waermepumpe.html

radiator_investment_range = [550, 1150] # source: https://www.heizung.de/finanzielles/wissen/neue-heizkoerper-kosten-und-foerdermittel.html

# single family home may have only 6 radiators but they are typically larger and priced at the higher end
n_radiator_single_home = 6 
# in multi family buildings the number may be 5 per floor but they are smaller and priced on the lower end.
n_radiator_multi_home = 5*3
n_radiators_cost = [n_radiator_single_home*radiator_investment_range[1], n_radiator_multi_home*radiator_investment_range[0]]
subvention = 0.7

regular_heatpump_renovation_investment_min = (heatpump_investment_range[0]+n_radiators_cost[0])*(1-subvention)
regular_heatpump_renovation_investment_max = (heatpump_investment_range[1]+n_radiators_cost[1])*(1-subvention)
regular_heatpump_renovation_investment_range = [regular_heatpump_renovation_investment_min, regular_heatpump_renovation_investment_max]
improved_heatpump_stock_home_investment_range = [heatpump_investment_range[0]*(1-subvention), heatpump_investment_range[1]*(1-subvention)]	

# ratio of stock_home to new homes in 2022
# Source: https://www.waermepumpe.de/fileadmin/user_upload/waermepumpe/08_Sonstige/Filedump/BWP_Branchenstudie_2023_DRUCK.pdf page 11
total_sales_stock_homes = 175000
total_sales_new_homes = 62000
stock_home_to_new_homes_ratio = total_sales_stock_homes / total_sales_new_homes

In [15]:
# calculate heat demand in kWh for a given number of heating degree days
def monthly_heat_demand_from_hdd(hdd, annual_heat_demand):
    return (hdd / sum(monthly_hdd)) * annual_heat_demand


def calculate_monthly_cop(study:HeatPumpStudy, heating_temp, avg_ambient_temp, monthly_heat_demand):
    # convert monthly heat demand from kWh to W assuming 100% duty cycle and 30 days per month
    study.Q_out = monthly_heat_demand / (24 * 30)*1000
    study.setup_network(iterinfo=False)
    heating_temp = max(avg_ambient_temp+20, heating_temp) # heating temp is at least 20 degrees above ambient temperature to account for minimal temperature lift
    study.set_boundary_conditions(T_cond=heating_temp+dT_condenser, T_evap=avg_ambient_temp-dT_evaporator)
    study.solve()
    return study.calculate_cop()

def calculate_monthly_co2_emissions(energy_mix, cop, heat_demand_kwh):
    co2_emissions_per_kwh = (
        energy_mix["coal"] * coal_co2_emissions
        + energy_mix["natural_gas"] * natural_gas_co2_emissions
        + energy_mix["nuclear"] * nuclear_co2_emissions
        + energy_mix["renewable"] * renewable_co2_emissions
    )
    return (heat_demand_kwh / cop) * co2_emissions_per_kwh

def calculate_monthly_energy_consumption(cop, heat_demand_kwh):
    return heat_demand_kwh / cop

def calculate_annual_co2_emissions(heatpump:HeatPumpStudy, annual_heating_demand, heating_temp):
    annual_co2_emissions = 0
    for i in range(12):
        monthly_heat_demand = monthly_heat_demand_from_hdd(monthly_hdd[i], annual_heating_demand)
        avg_ambient_temp = monthly_ambient_temps[i]
        cop = calculate_monthly_cop(heatpump,heating_temp, avg_ambient_temp, monthly_heat_demand)
        co2_emissions = calculate_monthly_co2_emissions(monthly_energy_mix[i], cop, monthly_heat_demand)
        #print(f"CO2 emissions for month {str(i+1)}: {str(co2_emissions)} kg")
        annual_co2_emissions += co2_emissions
    return annual_co2_emissions

def calculate_annual_energy_consumption(heatpump:HeatPumpStudy, annual_heating_demand, heating_temp):
    annual_energy_consumption = 0
    for i in range(12):
        monthly_heat_demand = monthly_heat_demand_from_hdd(monthly_hdd[i], annual_heating_demand)
        avg_ambient_temp = monthly_ambient_temps[i]
        cop = calculate_monthly_cop(heatpump,heating_temp, avg_ambient_temp, monthly_heat_demand)
        energy_consumption = calculate_monthly_energy_consumption(cop, monthly_heat_demand)
        #print(f"Energy consumption for month {str(i+1)}: {str(energy_consumption)} kWh")
        annual_energy_consumption += energy_consumption
    return annual_energy_consumption

In [ ]:
def main():
    #Sales prognosis for 2026, 2027
    total_sales = [3600, 12700]
    # yearly growth of 30% beyond that
    yearly_growth = 0.3
    total_sales.extend([round(total_sales[-1] * (1 + yearly_growth) ** i) for i in range(1, 10)])
    years=len(total_sales)

    new_home_heatpumps = [round(total_sales[i] / (1 + stock_home_to_new_homes_ratio)) for i in range(years)]
    stock_home_heatpumps = [round(total_sales[i] /(1+1/ stock_home_to_new_homes_ratio)) for i in range(years)]

    regular_heatpump=RegularHeatPumpStudy()
    regular_heatpump_new_home_annual_energy = calculate_annual_energy_consumption(regular_heatpump, new_home_heating_demand, new_home_heating_temp)
    regular_heatpump_new_home_annual_co2 = calculate_annual_co2_emissions(regular_heatpump, new_home_heating_demand, new_home_heating_temp)
    regular_heatpump_new_home_annual_energy_cost = regular_heatpump_new_home_annual_energy * electricity_cost
    regular_heatpump_stock_home_annual_energy = calculate_annual_energy_consumption(regular_heatpump, stock_home_heating_demand, stock_home_heating_temp)
    regular_heatpump_stock_home_annual_co2 = calculate_annual_co2_emissions(regular_heatpump, stock_home_heating_demand, stock_home_heating_temp)
    regular_heatpump_stock_home_annual_energy_cost = regular_heatpump_stock_home_annual_energy * electricity_cost
    regular_heatpump_revonation_annual_energy = calculate_annual_energy_consumption(regular_heatpump, stock_home_heating_demand, renovated_heating_temp)
    regular_heatpump_renovation_annual_co2 = calculate_annual_co2_emissions(regular_heatpump, stock_home_heating_demand, renovated_heating_temp)
    regular_heatpump_renovation_annual_energy_cost = regular_heatpump_revonation_annual_energy * electricity_cost
    print(f"regular heat pump stock home annual energy cost",regular_heatpump_stock_home_annual_energy_cost)
    print(f"regular heat pump rennovation annual energy cost",regular_heatpump_renovation_annual_energy_cost)
    
    improved_heatpump=InternalCondenserHeatPumpStudy(expansion_device="expander", N=1)
    improved_heatpump_new_home_annual_energy = calculate_annual_energy_consumption(improved_heatpump, new_home_heating_demand, new_home_heating_temp)
    improved_heatpump_new_home_annual_co2 = calculate_annual_co2_emissions(improved_heatpump, new_home_heating_demand, new_home_heating_temp)
    improved_heatpump_new_home_annual_energy_cost = improved_heatpump_new_home_annual_energy * electricity_cost
    improved_heatpump_stock_home_annual_energy = calculate_annual_energy_consumption(improved_heatpump, stock_home_heating_demand, stock_home_heating_temp)
    improved_heatpump_stock_home_annual_co2 = calculate_annual_co2_emissions(improved_heatpump, stock_home_heating_demand, stock_home_heating_temp)
    improved_heatpump_stock_home_annual_energy_cost = improved_heatpump_stock_home_annual_energy * electricity_cost
    print(f"improved heat pump stock home annual energy cost",improved_heatpump_stock_home_annual_energy_cost)
    gas_furnace_new_home_annual_energy_cost = new_home_heating_demand * gas_cost
    gas_furnace_stock_home_annual_energy_cost = stock_home_heating_demand * gas_cost
    

    relative_energy_savings_new_home = (1-improved_heatpump_new_home_annual_energy/regular_heatpump_new_home_annual_energy)*100
    relative_co2_savings_new_home = (1-improved_heatpump_new_home_annual_co2/regular_heatpump_new_home_annual_co2)*100
    relative_cost_savings_new_home = (1-improved_heatpump_new_home_annual_energy_cost/regular_heatpump_new_home_annual_energy_cost)*100
    relative_cost_savings_compared_to_gas_heating_new_home = (1-improved_heatpump_new_home_annual_energy_cost/gas_furnace_new_home_annual_energy_cost)*100
    relative_energy_savings_stock_home = (1-improved_heatpump_stock_home_annual_energy/regular_heatpump_stock_home_annual_energy)*100
    relative_co2_savings_stock_home = (1-improved_heatpump_stock_home_annual_co2/regular_heatpump_stock_home_annual_co2)*100
    relative_cost_savings_stock_home = (1-improved_heatpump_stock_home_annual_energy_cost/regular_heatpump_stock_home_annual_energy_cost)*100
    relative_cost_savings_compared_to_gas_heating_stock_home = (1-improved_heatpump_stock_home_annual_energy_cost/gas_furnace_stock_home_annual_energy_cost)*100
    
    print(f"heat demand in new homes: {new_home_heating_demand} kWh")
    print(f"heat demand in stock_homes: {stock_home_heating_demand} kWh")
    print(f"cost for gas heating in new homes: {gas_furnace_new_home_annual_energy_cost} €")
    print(f"cost for gas heating in stock_homes: {gas_furnace_stock_home_annual_energy_cost} €")
    print("")
    print(f"relative energy savings in new homes: {round(relative_energy_savings_new_home,2)}%")
    print(f"relative co2 savings in new homes: {round(relative_co2_savings_new_home,2)}%")
    print(f"relative cost savings in new homes: {round(relative_cost_savings_new_home,2)}%")
    print(f"relative cost savings compared to gas heating in new homes: {round(relative_cost_savings_compared_to_gas_heating_new_home,2)}%")
    print(f"relative energy savings in stock_homes: {round(relative_energy_savings_stock_home,2)}%")
    print(f"relative co2 savings in stock_homes: {round(relative_co2_savings_stock_home,2)}%")
    print(f"relative cost savings in stock_homes: {round(relative_cost_savings_stock_home,2)}%")
    print(f"relative cost savings compared to gas heating in stock_homes: {round(relative_cost_savings_compared_to_gas_heating_stock_home,2)}%")
    print("")

    absolute_energy_savings_new_home = regular_heatpump_new_home_annual_energy-improved_heatpump_new_home_annual_energy
    absolute_co2_savings_new_home = regular_heatpump_new_home_annual_co2-improved_heatpump_new_home_annual_co2
    absolute_cost_savings_new_home = regular_heatpump_new_home_annual_energy_cost-improved_heatpump_new_home_annual_energy_cost
    absolute_cost_savings_compared_to_gas_heating_new_home = gas_furnace_new_home_annual_energy_cost-improved_heatpump_new_home_annual_energy_cost
    absolute_energy_savings_stock_home = regular_heatpump_stock_home_annual_energy-improved_heatpump_stock_home_annual_energy
    absolute_co2_savings_stock_home = regular_heatpump_stock_home_annual_co2-improved_heatpump_stock_home_annual_co2
    absolute_cost_savings_stock_home = regular_heatpump_stock_home_annual_energy_cost-improved_heatpump_stock_home_annual_energy_cost
    absolute_cost_savings_compared_to_gas_heating_stock_home = gas_furnace_stock_home_annual_energy_cost-improved_heatpump_stock_home_annual_energy_cost
    absolute_cost_savings_compared_to_gas_heating_renovation = gas_furnace_stock_home_annual_energy_cost-regular_heatpump_renovation_annual_energy_cost
    print(f"absolute energy savings in new homes: {round(absolute_energy_savings_new_home,2)} kWh")
    print(f"absolute co2 savings in new homes: {round(absolute_co2_savings_new_home,2)} kg")
    print(f"absolute cost savings in new homes: {round(absolute_cost_savings_new_home,2)} €")
    print(f"absolute cost savings compared to gas heating in new homes: {round(absolute_cost_savings_compared_to_gas_heating_new_home,2)} €")
    print(f"absolute energy savings in stock_homes: {round(absolute_energy_savings_stock_home,2)} kWh")
    print(f"absolute co2 savings in stock_homes: {round(absolute_co2_savings_stock_home,2)} kg")
    print(f"absolute cost savings in stock_homes: {round(absolute_cost_savings_stock_home,2)} €")
    print(f"absolute cost savings compared to gas heating in stock_homes: {round(absolute_cost_savings_compared_to_gas_heating_stock_home,2)} €")
    print(f"absolute cost savings compared to gas heating in renovation: {round(absolute_cost_savings_compared_to_gas_heating_renovation,2)} €")
    print(f"average energy savings: {round((absolute_energy_savings_new_home*total_sales_new_homes+absolute_energy_savings_stock_home*total_sales_stock_homes)/(total_sales_new_homes+total_sales_stock_homes),2)} kWh")
    print(f"average co2 savings: {round((absolute_co2_savings_new_home*total_sales_new_homes+absolute_co2_savings_stock_home*total_sales_stock_homes)/(total_sales_new_homes+total_sales_stock_homes),2)} kg")
    print(f"average cost savings: {round((absolute_cost_savings_new_home*total_sales_new_homes+absolute_cost_savings_stock_home*total_sales_stock_homes)/(total_sales_new_homes+total_sales_stock_homes),2)} €")

    renovation_time_to_roi=regular_heatpump_renovation_investment_range/(absolute_cost_savings_compared_to_gas_heating_renovation)
    print(f"renovation time to roi:{renovation_time_to_roi}")
    improved_heatpump_time_to_roi=improved_heatpump_stock_home_investment_range/(absolute_cost_savings_compared_to_gas_heating_stock_home)
    print(f"imprved heatpump time to roi:{improved_heatpump_time_to_roi}")
    # let's export those to a dataframe and then to an excel sheet
    single_home_results = [
        [
            "New homes",
            regular_heatpump_new_home_annual_energy,
            regular_heatpump_new_home_annual_co2,
            improved_heatpump_new_home_annual_energy,
            improved_heatpump_new_home_annual_co2,
            relative_energy_savings_new_home,
            relative_co2_savings_new_home,
            relative_cost_savings_new_home,
            relative_cost_savings_compared_to_gas_heating_new_home,
            absolute_energy_savings_new_home,
            absolute_co2_savings_new_home,
            absolute_cost_savings_new_home,
            absolute_cost_savings_compared_to_gas_heating_new_home,
            specific_new_home_heating_demand,
            heating_area,
            new_home_heating_demand,
            new_home_heating_temp,
            electricity_cost,
            gas_cost,
            
        ],
        [
            "stock_homes",
            regular_heatpump_stock_home_annual_energy,
            regular_heatpump_stock_home_annual_co2,
            improved_heatpump_stock_home_annual_energy,
            improved_heatpump_stock_home_annual_co2,
            relative_energy_savings_stock_home,
            relative_co2_savings_stock_home,
            relative_cost_savings_stock_home,
            relative_cost_savings_compared_to_gas_heating_stock_home,
            absolute_energy_savings_stock_home,
            absolute_co2_savings_stock_home,
            absolute_cost_savings_stock_home,
            absolute_cost_savings_compared_to_gas_heating_stock_home,
            specific_stock_home_heating_demand,
            heating_area,
            stock_home_heating_demand,
            stock_home_heating_temp,
            
        ],
    ]
    # round all values to 2 decimal places
    for i in range(len(single_home_results)):
        for j in range(1, len(single_home_results[i])):
            single_home_results[i][j] = round(single_home_results[i][j], 2)
            
    
    
    # let's add titles to each row
    rows = ["",
               "regular Heatpump energy consumption [kWh/year]", 
               "regular Heatpump CO2 emissions [kg/year]", 
               "improved Heatpump energy consumption [kWh/year]", 
               "improved Heatpump CO2 emissions [kg/year]", 
               "relative energy savings [%]", 
               "relative CO2 savings [%]", 
               "relative cost savings [%]",
               "relative savings compared to gas heating [%]",
               "absolute energy savings [kWh/year]", 
               "absolute CO2 savings [kg/year]",                                  
               "absolute cost savings [€/year]",
               "absolute savings compared to gas heating [€/year]"]
    
    # let's add a few rows containing the relevant assumptions regarding price, heating demand, heating area, etc.
    rows.extend(["specific heating demand [kWh/year]",
                 "heating area [m²]",
                 "heat demand [kWh/year]",
                 "heating temperature [°C]",
                 "electricity_cost [€/kWh_el]",
                 "gas_cost [€/kWh_th]",
    ])
                 
                    
    
    single_home_results.insert(0,rows)
    # now we create a dataframe and export it to excel
    single_home_results_df = pd.DataFrame(single_home_results)
    # this produces a sheet which has rows and columns swapped so let's transpose it
    single_home_results_df = single_home_results_df.transpose()
    single_home_results_df.to_excel("output/single-home-results.xlsx", index=False)

    results = [
        [
            year,
            new_home_heatpumps[i],
            stock_home_heatpumps[i],
            regular_heatpump_new_home_annual_energy * new_home_heatpumps[i] + regular_heatpump_stock_home_annual_energy * stock_home_heatpumps[i],
            regular_heatpump_new_home_annual_co2 * new_home_heatpumps[i] + regular_heatpump_stock_home_annual_co2 * stock_home_heatpumps[i],
            improved_heatpump_new_home_annual_energy * new_home_heatpumps[i] + improved_heatpump_stock_home_annual_energy * stock_home_heatpumps[i],
            improved_heatpump_new_home_annual_co2 * new_home_heatpumps[i] + improved_heatpump_stock_home_annual_co2 * stock_home_heatpumps[i],
            ((regular_heatpump_new_home_annual_energy - improved_heatpump_new_home_annual_energy) * new_home_heatpumps[i] + (regular_heatpump_stock_home_annual_energy - improved_heatpump_stock_home_annual_energy) * stock_home_heatpumps[i])/1e6,
            ((regular_heatpump_new_home_annual_co2 - improved_heatpump_new_home_annual_co2) * new_home_heatpumps[i] + (regular_heatpump_stock_home_annual_co2 - improved_heatpump_stock_home_annual_co2) * stock_home_heatpumps[i])/1e3,
        ]
        for year, i in enumerate(range(years), start=2026)
    ]

    # we want to calculate the cumulative savings saved that year.
    # since the installed heat pumps continue to provide savings in the coming years, 
    # we need to sum up the savings from the previous years

    results_cumulative=results.copy()
    for i in range(1, len(results)):
        results_cumulative[i][3] = sum(results[j][3] for j in range(i + 1))
        results_cumulative[i][4] = sum(results[j][4] for j in range(i + 1))
        results_cumulative[i][5] = sum(results[j][5] for j in range(i + 1))
        results_cumulative[i][6] = sum(results[j][6] for j in range(i + 1))
        results_cumulative[i][7] = sum(results[j][7] for j in range(i + 1))
        results_cumulative[i][8] = sum(results[j][8] for j in range(i + 1))

    #store results to excel sheet in ouput/results.xlsx
    results_df = pd.DataFrame(
        results_cumulative,
        columns=[
            "Year",
            "New homes",
            "stock_homes",
            "Regular energy consumption [kWh]",
            "Regular CO2 emissions [kg]",
            "Improved energy consumption",
            "Improved CO2 emissions",
            "Energy savings [GWh]",
            "CO2 savings [Tons]",
        ],
    )
    
    
    results_df.to_excel("output/cumulative-benefit-forecast.xlsx", index=False)
    
    

In [ ]:
main()

regular heat pump stock home annual energy cost 2574.313120448093
regular heat pump rennovation annual energy cost 1709.7629199770083
improved heat pump stock home annual energy cost 2014.6013472943923
heat demand in new homes: 5490 kWh
heat demand in stock_homes: 20130 kWh
cost for gas heating in new homes: 625.86 €
cost for gas heating in stock_homes: 2294.82 €

relative energy savings in new homes: 12.86%
relative co2 savings in new homes: 12.86%
relative cost savings in new homes: 12.86%
relative cost savings compared to gas heating in new homes: 42.12%
relative energy savings in stock_homes: 21.74%
relative co2 savings in stock_homes: 21.75%
relative cost savings in stock_homes: 21.74%
relative cost savings compared to gas heating in stock_homes: 12.21%

absolute energy savings in new homes: 152.75 kWh
absolute co2 savings in new homes: 57.64 kg
absolute cost savings in new homes: 53.46 €
absolute cost savings compared to gas heating in new homes: 263.6 €
absolute energy savings i

In [ ]:
import numpy as np
# Crankcase heater power rating (W)
crankcase_heater_power = 70

# Assumption: Crankcase heater operates when the ambient temperature is below 10°C
heater_operating_hours_per_day = [6 if temp < 10 else 0 for temp in monthly_ambient_temps]

monthly_hdd = [max(base_temperature - temp, 0) for temp in monthly_ambient_temps]

# Total number of days the crankcase heater operates in a year
operating_days_per_month = np.array(heater_operating_hours_per_day) * np.array(list(monthly_hdd))
total_operating_days = sum(operating_days_per_month)

# Energy wasted on crankcase heaters in a year (kWh)
energy_wasted = crankcase_heater_power * total_operating_days / 1000

## Assumptions for national CO2 savings calculation

- All buildings in Germany are assumed to be equipped with heat pumps (regular or with expander).
- Distribution of homes by required heating water temperature: 30°C: 13%, 40°C: 10%, 50°C: 30%, 60°C: 37%, 70°C: 10%.
- Building distribution (specific heating demand [kWh/m²] and percentage) is read from './source_data/gebaeudedaten-heizenergieverbrauch_allgemein_verteilung-bundesweit.xlsx', starting at row 4 (zero-based index 3), with kWh/m² in column A and percentage in column B.
- Assignment to temperature buckets is done by cumulative area share: the first 13% of the cumulative distribution goes to 30°C, the next 10% to 40°C, etc. This uses cumulative percentages rather than fixed counts so the slices reflect area share.
- Total livable area in Germany (2023) = 3,898,730,000 m². For each bucket we compute: area_in_bucket * weighted_average_specific_demand_in_bucket = total heating demand (kWh).
- The total heating demand per bucket is then passed to `calculate_annual_co2_emissions(...)` to obtain scaled CO2 emissions.


In [17]:
import pandas as pd

# Read the building data from the Excel file
df = pd.read_excel('./source_data/gebaeudedaten-heizenergieverbrauch_allgemein_verteilung-bundesweit.xlsx', header=None)
sheet = df

# Read data starting from row 4 (index 3)
specific_demands = []
percentages = []
for row in range(3, len(sheet)):
    kwh_per_m2 = sheet.iloc[row, 0]
    percent = sheet.iloc[row, 1]
    if pd.notnull(kwh_per_m2) and pd.notnull(percent):
        specific_demands.append(float(kwh_per_m2))
        percentages.append(float(percent))

# Normalize percentages to sum to 1
percentages = [p / 100 for p in percentages]
total_area = 3898730000  # m²

# Map temperature buckets to their shares
temp_buckets = [
    (30, 0.13),
    (40, 0.10),
    (50, 0.30),
    (60, 0.37),
    (70, 0.10)
]

# Assign each bucket a slice of the distribution using cumulative percentages
# Build cumulative list of the percentages (they already sum to 1 after normalization)
cum_list = []
running = 0.0
for p in percentages:
    running += p
    cum_list.append(running)

# thresholds are the cumulative shares where each bucket should end
thresholds = []
cum = 0.0
for _, share in temp_buckets:
    cum += share
    thresholds.append(cum)

bucket_indices = []
start = 0
for t in thresholds:
    # find smallest index where cumulative percentage >= t
    end = next((i+1 for i, c in enumerate(cum_list) if c >= t), len(percentages))
    bucket_indices.append((start, end))
    start = end

# ensure last bucket ends at the end of the distribution
if bucket_indices:
    bucket_indices[-1] = (bucket_indices[-1][0], len(percentages))

# Calculate total heating demand for each bucket
bucket_heating_demands = []
for (temp, _), (start, end) in zip(temp_buckets, bucket_indices):
    area_share = sum(percentages[start:end])
    area = area_share * total_area
    # Weighted average specific demand for this bucket
    if end > start and area_share > 0:
        avg_specific_demand = sum(
            specific_demands[i] * percentages[i] for i in range(start, end)
        ) / area_share
    else:
        avg_specific_demand = 0
    total_demand = area * avg_specific_demand
    bucket_heating_demands.append((temp, total_demand))

# Calculate CO2 emissions for each bucket for both heat pump types
regular_hp = RegularHeatPumpStudy()
improved_hp = RegularHeatPumpStudy(expansion_device="expander")

results = []
for temp, demand in bucket_heating_demands:
    co2_regular = calculate_annual_co2_emissions(regular_hp, demand, temp)
    co2_improved = calculate_annual_co2_emissions(improved_hp, demand, temp)
    results.append({
        "heating_temp": temp,
        "total_heating_demand_kWh": demand,
        "co2_regular_hp_kg": co2_regular,
        "co2_improved_hp_kg": co2_improved,
        "co2_savings_kg": co2_regular - co2_improved
    })

# Summing up for all of Germany
total_demand = sum(r["total_heating_demand_kWh"] for r in results)
total_co2_regular = sum(r["co2_regular_hp_kg"] for r in results)
total_co2_improved = sum(r["co2_improved_hp_kg"] for r in results)
total_co2_savings = total_co2_regular - total_co2_improved

# --- Baseline (today) heating CO2 calculation ---
# BAFA emission factors (kg CO2 per kWh_th)
bafa = {
    'heizoel_leicht': 0.266,
    'erdgas': 0.201,
    'holzpellets': 0.036,
    'biogas_biomethan': 0.152,
    'braunkohle': 0.383,
}

# Heating type shares (percent) from BDEW-like source (map to category averages)
# These numbers represent percent of heated dwellings using each heating type
heating_type_shares = {
    'Gas-Zentralheizung': 42.2,
    'Öl-Zentralheizung': 19.1,
    'Fernwärme': 15.4,
    'Gas-Etagenheizung': 9.0,
    'Holz_Pellet': 3.6,
    'Elektro_WP': 3.5,
    'Sonstige': 3.4,
    'Stromspeicher': 2.5,
}

# Map heating types to representative fuel factors (kg CO2 per kWh_th)
# Assume Fernwärme and Sonstige use mixed sources; use approximate averages where needed
heating_type_factors = {
    'Gas-Zentralheizung': bafa['erdgas'],
    'Gas-Etagenheizung': bafa['erdgas'],
    'Öl-Zentralheizung': bafa['heizoel_leicht'],
    'Holz_Pellet': bafa['holzpellets'],
    'Elektro_WP': None,  # electric-driven — compare using grid intensity below
    'Fernwärme': None,  # handled separately as mixed source
    'Sonstige': None,
    'Stromspeicher': None,
}

# Compute average grid CO2 intensity (kg CO2 per kWh_el) from monthly_energy_mix (array of dicts)
# monthly_energy_mix is a list of dicts with keys: 'coal','natural_gas','nuclear','renewable' representing shares
grid_monthly_intensities = []
for mix in monthly_energy_mix:
    intensity = (mix['coal'] * coal_co2_emissions + mix['natural_gas'] * natural_gas_co2_emissions + mix['nuclear'] * nuclear_co2_emissions + mix['renewable'] * renewable_co2_emissions)
    grid_monthly_intensities.append(intensity)
avg_grid_intensity = sum(grid_monthly_intensities) / len(grid_monthly_intensities)

# --- Improved baseline factor assignments ---
# Fernwärme average intensity from data (198 g/kWh = 0.198 kg/kWh)
heating_type_factors['Fernwärme'] = 0.198

# For electric heating stock: compute the regular heat pump intensity per kWh_th (kg CO2 per kWh_th)
# as total_co2_regular / total_demand if available. This captures the real modelled regular-HP emissions.
reg_hp_intensity_per_kwh_th = None
if total_demand > 0:
    reg_hp_intensity_per_kwh_th = total_co2_regular / total_demand

# If reg_hp_intensity_per_kwh_th is missing or non-positive, fall back to assuming COP=3
if not reg_hp_intensity_per_kwh_th or reg_hp_intensity_per_kwh_th <= 0:
    fallback_cop = 3.0
    reg_hp_intensity_per_kwh_th = avg_grid_intensity / fallback_cop

# Use this as the proxy for existing electric heating (Elektro_WP)
heating_type_factors['Elektro_WP'] = reg_hp_intensity_per_kwh_th

# Sonstige: approximate as weighted average of oil and gas (proxy for mixed small-scale systems)
heating_type_factors['Sonstige'] = 0.5 * (bafa['heizoel_leicht'] + bafa['erdgas'])
# Stromspeicher (electric storage heaters): use avg grid intensity directly (kg CO2/kWh_th ~= kg CO2/kWh_el)
heating_type_factors['Stromspeicher'] = avg_grid_intensity

# Compute weighted baseline CO2 per kWh_th using heating_type_shares and heating_type_factors
total_share = sum(heating_type_shares.values())
baseline_co2_per_kwh_th = 0.0
for ht, share in heating_type_shares.items():
    weight = share / total_share
    factor = heating_type_factors.get(ht)
    if factor is None:
        # last-resort fallback: use avg_grid_intensity
        factor = avg_grid_intensity
    baseline_co2_per_kwh_th += weight * factor

# Now compute national baseline CO2 assuming total_demand (kWh) is the building heat demand computed earlier
# total_demand is sum of bucket heating demands (kWh), which we computed above
national_baseline_co2_kg = total_demand * baseline_co2_per_kwh_th

# Compare baseline to regular/improved heat pump totals
print("--- Baseline heating CO2 (today) and comparisons ---")
print("Baseline average CO2 per kWh_th (kg):", round(baseline_co2_per_kwh_th, 4))
print("National baseline heating CO2 (tons):", round(national_baseline_co2_kg / 1000, 2))
print("Total CO2 with regular heat pumps (tons):", round(total_co2_regular / 1000, 2))
print("Total CO2 with improved (expander) heat pumps (tons):", round(total_co2_improved / 1000, 2))
print("Absolute CO2 savings vs today (regular hp) (tons):", round((national_baseline_co2_kg - total_co2_regular) / 1000, 2))
print("Absolute CO2 savings vs today (improved hp) (tons):", round((national_baseline_co2_kg - total_co2_improved) / 1000, 2))
print("Relative savings vs today (regular hp) (%):", round(100*(national_baseline_co2_kg - total_co2_regular)/national_baseline_co2_kg,2))
print("Relative savings vs today (improved hp) (%):", round(100*(national_baseline_co2_kg - total_co2_improved)/national_baseline_co2_kg,2))



--- Baseline heating CO2 (today) and comparisons ---
Baseline average CO2 per kWh_th (kg): 0.2097
National baseline heating CO2 (tons): 127860647.81
Total CO2 with regular heat pumps (tons): 71336792.65
Total CO2 with improved (expander) heat pumps (tons): 57805193.0
Absolute CO2 savings vs today (regular hp) (tons): 56523855.16
Absolute CO2 savings vs today (improved hp) (tons): 70055454.81
Relative savings vs today (regular hp) (%): 44.21
Relative savings vs today (improved hp) (%): 54.79
